# Cameo Requirements Extraction + Tag Update — Chained Jobs (SDK only)

A recipe for chaining jobs on the Istari Digital Platform from Python using the official [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup) SDK directly — no `istari_fluent` wrapper.

The same workflow as the companion notebook: register a Cameo `.mdzip` file as a Model, run an extraction job to read requirements, then chain a tag update job to write `Selected part number: PN12345` back into the model.

### What we cover

- Connecting to the platform with a Personal Access Token.
- Registering a Cameo `.mdzip` file as a Model via `client.add_model()`.
- Running a first extraction job with `client.add_job()` and polling for completion.
- Reading products directly from the job's file revision.
- Chaining a second job to update a tag using `@istari:update_tags`.
- Tracing the backward lineage of the final result.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**. See the [Sign-up Guide](https://docs.istaridigital.com/users/account/sign-up) and [Personal Access Tokens](https://docs.istaridigital.com/users/user-guide/settings#developer-settings--personal-access-tokens).
- An agent with the **Cameo** integration and access to `@istari:extract` and `@istari:update_tags`. See [Manage Tool Access](https://docs.istaridigital.com/users/admin-guide/user-management#manage-tool-access-for-a-user).
- A Cameo `.mdzip` requirements file to extract from.

### 1 &middot; Credentials

Create a `.env` file **next to this notebook** with:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

### 2 &middot; Install dependencies

```bash
pip install istari-digital-client python-dotenv
```

## 1 &middot; Connect and verify

Read credentials from `.env`, construct the `Client`, and run a readiness check.

In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from istari_digital_client import Client, Configuration, JobStatusName

load_dotenv(".env")

registry_url = os.environ["ISTARI_REGISTRY_URL"]
token = os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"]

client = Client(Configuration(registry_url=registry_url, registry_auth_token=token))

# Corporate network with an internal CA bundle? Set REQUESTS_CA_BUNDLE and SSL_CERT_FILE first:
# os.environ["REQUESTS_CA_BUNDLE"] = "/path/to/ca.pem"
# os.environ["SSL_CERT_FILE"] = "/path/to/ca.pem"

report = client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(f"Connected to {registry_url}")

## 2 &middot; Configure job parameters

In [ ]:
# --- Configure these for your environment ---
MDZIP_PATH = Path.cwd() / "NCXTable-example.mdzip"  # path to your Cameo file
TOOL_VERSION = "2024x-refresh2"                             # adjust to your Cameo version
OPERATING_SYSTEM = "Windows 11"                             # adjust to your agent's OS
DISPLAY_NAME = "NCXTable-example.mdzip"
EXTERNAL_ID = "cameo-requirements-extraction-tutorial-ncxtable-demo"
# --------------------------------------------

assert MDZIP_PATH.exists(), f"Cameo file not found: {MDZIP_PATH}"
print(f"Using model file: {MDZIP_PATH}")
print(f"Tool version:     {TOOL_VERSION}")
print(f"Operating system: {OPERATING_SYSTEM}")

## 3 &middot; Register the Cameo file as a Model Resource

`client.add_model()` uploads the local file to the platform and registers it as a **Model** resource.

In [ ]:
model = client.add_model(
    path=MDZIP_PATH,
    external_identifier=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
model_id = model.id
print(f"Uploaded new model id={model_id}")

# The model's file and its latest revision
model_file = model.file
model_rev = model_file.revisions[-1] if model_file and model_file.revisions else None
print(f"file_id={model_file.id if model_file else None}  rev_id={model_rev.id if model_rev else None}")

## 4 &middot; Helper: poll a job until it reaches a terminal state

The SDK's `client.add_job()` returns immediately — you poll `client.get_job(id)` until the status is `COMPLETED` or `FAILED`.

In [ ]:
def wait_for_job(job_id: str, *, timeout: int = 600, poll_interval: int = 5):
    """Poll until terminal state. Returns the final Job object."""
    start = time.time()
    while True:
        job = client.get_job(job_id)
        status_name = job.status.name
        print(f"  [{status_name.value if hasattr(status_name, 'value') else status_name}] id={job.id}")

        if status_name == JobStatusName.PENDING:
            msg = getattr(job.status, "message", None)
            if msg and "None agent" in msg:
                raise RuntimeError(f"No agent available: {msg}")

        if status_name in (JobStatusName.COMPLETED, JobStatusName.FAILED):
            return job

        if time.time() - start > timeout:
            raise TimeoutError(f"Job {job_id} did not finish within {timeout}s")

        time.sleep(poll_interval)


def require_completed(job):
    if job.status.name != JobStatusName.COMPLETED:
        raise RuntimeError(f"Job {job.id} ended with status {job.status.name}")
    return job


def get_job_products(job):
    """Return the list of Product records from the job's latest file revision."""
    if not job.file or not job.file.revisions:
        return []
    rev = job.file.revisions[-1]
    return rev.products or []


def find_product_by_filename(job, filename: str):
    """Return the (Product, FileRevision) pair whose revision name matches filename."""
    for p in get_job_products(job):
        if p.revision_id:
            rev = client.get_revision(p.revision_id)
            if rev.name == filename:
                return p, rev
    return None, None


print("Helpers defined.")

## 5 &middot; Run the first extraction job

`client.add_job()` queues the job. We then poll until it reaches a terminal state.

In [ ]:
job1_raw = client.add_job(
    model_id=model_id,
    function="@istari:extract",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
)
print(f"Submitted job {job1_raw.id}; polling...")

job1 = require_completed(wait_for_job(job1_raw.id))
print(f"\nJob 1 finished: {job1.status.name}")

## 6 &middot; Inspect the products

The job's latest file revision records every artifact it wrote as a `Product`.
Each `Product` carries a `resource_id` and a `revision_id` pointing to the exact artifact revision the agent produced.

In [ ]:
products_1 = get_job_products(job1)
print(f"Job 1 wrote {len(products_1)} product(s):\n")

for p in products_1:
    rev = client.get_revision(p.revision_id) if p.revision_id else None
    name = rev.name if rev else "?"
    rtype = p.resource_type or "?"
    print(f"  - {rtype:10s}  name={name!r:40s}  file={getattr(rev, 'file_id', '?')}  rev={p.revision_id}")

## 7 &middot; Read requirements and pick one to update

Load `requirements.json` from Job 1 by reading the artifact's content bytes via `client.read_contents()`.

In [ ]:
TARGET_REQUIREMENT_NAME = "Requirement REQ-001 Plate Thickness"  # adjust to match a requirement in your model

req_product, req_rev = find_product_by_filename(job1, "requirements.json")
assert req_rev is not None, (
    "requirements.json not found among job products. "
    f"Available: {[client.get_revision(p.revision_id).name for p in products_1 if p.revision_id]}"
)

requirements_data = json.loads(client.read_contents(token=req_rev.content_token).decode("utf-8"))
print(f"Found {len(requirements_data)} requirements\n")

for req in requirements_data:
    print(f"  id={req['id']}")
    print(f"  name={req['name']}")
    print(f"  tags={req.get('tags', {})}")
    print()

target_req = next(
    (r for r in requirements_data if r["name"] == TARGET_REQUIREMENT_NAME),
    None
)
assert target_req is not None, (
    f"Requirement {TARGET_REQUIREMENT_NAME!r} not found. "
    f"Available names: {[r['name'] for r in requirements_data]}"
)

TARGET_ELEMENT_ID = target_req["id"]
print(f"\nTarget requirement:")
print(f"  name: {target_req['name']}")
print(f"  id:   {TARGET_ELEMENT_ID}")
print(f"  existing tags: {target_req.get('tags', {})}")

## 8 &middot; Chain a second job — update the tag

Run `@istari:update_tags` on the original `.mdzip` model to write `PN12345` back into the requirement.

The `parameters` dict is passed directly to `client.add_job()`. The result is a **new revision of the same model**.

In [ ]:
updates = [
    {
        "element_id": TARGET_ELEMENT_ID,
        "replace_existing": False,
        "tags": {
            "Text": "Selected part number: PN12345"
        }
    }
]

job2_raw = client.add_job(
    model_id=model_id,
    function="@istari:update_tags",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
    parameters={"updates": updates},
)
print(f"Submitted tag update job {job2_raw.id}; polling...")

job2 = require_completed(wait_for_job(job2_raw.id))
print(f"\nJob 2 finished: {job2.status.name}")

products_2 = get_job_products(job2)
print(f"\nJob 2 wrote {len(products_2)} product(s):")
for p in products_2:
    rev = client.get_revision(p.revision_id) if p.revision_id else None
    name = rev.name if rev else "?"
    rtype = p.resource_type or "?"
    print(f"  - {rtype:10s}  name={name!r:40s}  rev={p.revision_id}")

## 9 &middot; Verify the update

Re-run extraction on the updated model revision, then read `requirements.json` to confirm the tag was written.

In [ ]:
job3_raw = client.add_job(
    model_id=model_id,
    function="@istari:extract",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
)
print(f"Submitted verification extraction {job3_raw.id}; polling...")

job3 = require_completed(wait_for_job(job3_raw.id))
print(f"\nVerification job finished: {job3.status.name}")

_, updated_req_rev = find_product_by_filename(job3, "requirements.json")
assert updated_req_rev is not None, "requirements.json not found in verification job products"

updated_requirements = json.loads(client.read_contents(token=updated_req_rev.content_token).decode("utf-8"))

updated_req = next(
    (r for r in updated_requirements if r["id"] == TARGET_ELEMENT_ID),
    None
)
assert updated_req is not None, "Could not find target requirement in updated extraction"

print(f"\nUpdated requirement:")
print(f"  name: {updated_req['name']}")
print(f"  tags: {updated_req.get('tags', {})}")

## 10 &middot; Trace the lineage

Walk backward from the verification `requirements.json` revision through its sources.
Each `FileRevision` has a `sources` list; each `Source` carries `resource_type`, `resource_id`, and `revision_id`.

The chain shows: **Artifact ← Job(@istari:extract) ← Model ← Job(@istari:update_tags) ← Model (original upload)**

In [ ]:
def print_lineage(revision_id: str, indent: int = 0, max_depth: int = 8, _seen: set | None = None):
    """Recursively print the backward lineage tree for a FileRevision."""
    if _seen is None:
        _seen = set()
    if revision_id in _seen or indent > max_depth:
        return
    _seen.add(revision_id)

    rev = client.get_revision(revision_id)
    prefix = "  " * indent
    created = rev.created.strftime("%Y-%m-%d %H:%M") if rev.created else "?"

    # Determine the resource type from the file
    resource_label = "Revision"
    function_label = ""
    if rev.file_id:
        try:
            f = client.get_file(rev.file_id)
            resource_label = getattr(f, "resource_type", None) or resource_label
            resource_id = getattr(f, "resource_id", None)
            if resource_label == "Job" and resource_id:
                job = client.get_job(resource_id)
                fn = getattr(getattr(job, "function", None), "name", None)
                if fn:
                    function_label = f" ({fn} {resource_id})"
        except Exception:
            pass

    name = rev.display_name or rev.name or revision_id
    print(f"{prefix}- {resource_label} {name!r}{function_label}  rev={revision_id}  created={created}")

    sources = rev.sources or []
    # Prefer Job sources to keep the tree readable (mirrors fluent's behaviour)
    job_sources = [s for s in sources if getattr(s, "resource_type", None) == "Job"]
    promoted_sources = [s for s in sources if s.relationship_identifier == "promoted_from"]
    display_sources = job_sources + promoted_sources if job_sources else sources

    for src in display_sources:
        if src.revision_id:
            print_lineage(src.revision_id, indent + 1, max_depth, _seen)


print("Lineage for verification requirements.json:\n")
print_lineage(updated_req_rev.id)

## Verify in the UI

Sign in to the same platform you used for your token and cross-check:

1. **Files / Models** — You should see the model with the display name set in Step 3.
2. **Jobs / Activity** — Three jobs for `dassault_cameo`: two `@istari:extract` and one `@istari:update_tags`.
3. **Resources** — On the update_tags job, confirm the updated model revision was produced.
4. **Verification** — On the third job's resources, confirm `requirements.json` shows the part number `PN12345` in the target requirement.

## Optional &middot; Archive the model

Archiving hides the Model from default listings in the UI and API. The data and lineage stay intact — this is a soft delete you can reverse later.

In [ ]:
client.archive_model(model_id)